In [110]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from go_search_problem import GoProblem, GoState
from game_runner import GameRunner
import matplotlib.pyplot as plt
import pickle
from collections import deque
import itertools
from typing import List, Tuple, Dict
from open_spiel.python.rl_environment import Environment
from agents import RandomAgent, GreedyAgent, AlphaBetaAgent



print (torch.mps.is_available())


env = Environment("go", board_size=5)
state = env.reset()
    

True


# Defining q network

In [111]:
class DQN(nn.Module):
    def __init__(self, input_size, output_size):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(input_size, 70)
        self.layer2 = nn.Linear(70, 35)
        self.output_layer = nn.Linear(35, output_size)
        self.relu = nn.ReLU()


    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        return self.output_layer(x)

In [112]:
def save_model(path: str, model):
    """
    Save model to a file
    Input:
        path: path to save model to
        model: Pytorch model to save
    """
    torch.save({
        'model_state_dict': model.state_dict(),
    }, path)

def load_model(path: str, model):
    """
    Load model from file

    Note: you still need to provide a model (with the same architecture as the saved model))

    Input:
        path: path to load model from
        model: Pytorch model to load
    Output:
        model: Pytorch model loaded from file
    """
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    return model

In [113]:
def get_dqn_features(env, player):
    return env.observation_spec()["info_state"][player]

In [114]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, complete = zip(*batch)

        #convert to tensors
        return (torch.tensor(states, dtype=torch.float32),
                torch.tensor(actions, dtype=torch.int64),
                torch.tensor(rewards, dtype=torch.float32),
                torch.tensor(next_states, dtype=torch.float32),
                torch.tensor(complete, dtype=torch.float32))

    def buffer_size(self):
        return len(self.buffer)

In [115]:

gamma = 0.99
epsilon = 1
epsilon_min = 0.01
epsilon_decay = 0.995
learning_rate = 0.001
memory_size = 10000
batch_size = 64
# black is 0, white is 1
player = 0




In [116]:
def opponent_agents(episode, legal_actions, env):
     if episode < 100:
          return np.random.choice(legal_actions)
     elif episode < 500:
         agent = GreedyAgent()
     else:
        agent = AlphaBetaAgent()

     game_state = env.get_state
     return agent.get_move(game_state, time_limit=1)

In [ ]:
def train_DQN(env, num_episodes, learning_rate, board_size, memory_size, epsilon, player):

    env.reset()

    input_size = env.observation_spec()["info_state"][player]
    action_size = env.action_spec()["num_actions"]

    # train on device
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    policy_net = DQN(input_size, action_size)
    target_net = DQN(input_size, action_size)
    policy_net = policy_net.to(device)
    target_net = target_net.to(device)

    target_net.load_state_dict(policy_net.state_dict())


    optimizer = optim.Adam(policy_net.parameters(), lr=learning_rate)

    buffer = ReplayBuffer(memory_size)
    best_reward = float('-inf')

    

    # TODO: Implement Q-Learning
    for episode in range(num_episodes):
        total_reward = 0
        time_step = env.reset()


        while not time_step.last():
            # only train on our player's turn
            current_player = time_step.observations["current_player"]
            is_opening_move = time_step.first()

            if current_player == player:
                state = np.array(time_step.observations["info_state"][player])
                legal_actions = time_step.observations["legal_actions"][player]
                state_tensor = torch.tensor(state, dtype=torch.float32).to(device)

                if is_opening_move:
                    # hardcoded the opening move to be the center of 5x5 board.
                    action = 12


                # episilon-greedy 
                if np.random.rand() < epsilon:
                    action = np.random.choice(legal_actions)
                else:
                    q_vals = policy_net(state_tensor)

                    # masking illegal actions by setting their Q-values to -inf so they won't be chosen
                    mask = torch.full((26), float('-inf'))
                    mask[legal_actions] = 0

                    action = torch.argmax(q_vals + mask).item()

                next_time_step = env.step([action])
                reward = next_time_step.rewards[player]
                complete = next_time_step.last()

                next_state = np.array(next_time_step.observations["info_state"][player])
                buffer.push(state, action, reward, next_state, complete)


                # training
                if buffer.buffer_size() >= batch_size:
                    states, actions, rewards, next_states, complete = buffer.sample(batch_size)
                    states = states.to(device)
                    actions = actions.to(device)
                    rewards = rewards.to(device)
                    next_states = next_states.to(device)
                    complete = complete.to(device)

                    q_values = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
                    next_q_values = target_net(next_states).max(1)[0]
                    target_q_values = rewards + (gamma * next_q_values * (1 - complete))

                    loss = nn.MSELoss()(q_values, target_q_values.detach())
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                
                total_reward += reward
            
            # agents as the opponent
            else:
                legal_actions = time_step.observations["legal_actions"][current_player]
                #action = opponent_agents(episode, legal_actions, env)
                action = np.random.choice(legal_actions)
                next_time_step = env.step([action])
            
            time_step = next_time_step



        if episode % 50 == 0:
            target_net.load_state_dict(policy_net.state_dict())
            
            
            
        if total_reward > best_reward:
            best_reward = total_reward
            torch.save(policy_net.state_dict(), "dqn_model.pt")

    save_model("dqn_model.pt", policy_net)
    return policy_net


value_model = train_DQN(env, num_episodes=1000, learning_rate=learning_rate, board_size=5, memory_size=memory_size, epsilon=epsilon, player=player)

In [118]:
print(env.observation_spec()["info_state"][player])
print(env.action_spec()["num_actions"])

100
26


In [120]:
from heuristic_go_problems import GoProblemSimpleHeuristic, GoProblemLearnedHeuristic
from go_search_problem import HeuristicGoProblem

class DQNLearnedHeuristic(HeuristicGoProblem):
    def __init__(self, model=None, state=None):
        super().__init__(state=state)
        self.model = model


    def heuristic(self, state, player_index):
        feature = get_dqn_features(state, player_index)
        value = self.model(torch.tensor(feature, dtype=torch.float32)).item()

        return value

    def __str__(self) -> str:
        return "Learned Heuristic"
    



def create_value_agent_from_model():
    """
    Create agent object from saved model. This (or other methods like this) will be how your agents will be created in gradescope and in the final tournament.
    """

    model_path = "dqn_model.pt"
    # TODO: Update number of features for your own encoding size
    input_size = get_dqn_features(env, player)
    action_size = env.action_spec()["num_actions"]
    model = load_model(model_path, DQN(input_size, action_size))
    heuristic_search_problem = DQNLearnedHeuristic(model)

    # TODO: Try with other heuristic agents (IDS/AB/Minimax)
    learned_agent = GreedyAgent(heuristic_search_problem)

    return learned_agent

learned_agent = create_value_agent_from_model()
agent2 = GreedyAgent(GoProblemSimpleHeuristic())
print("Greedy Agent", agent2)
print("Learned Agent", learned_agent)

game_runner = GameRunner()
game_runner.play_tournament(learned_agent, agent2, num_games=100)


Greedy Agent GreedyAgent + Simple Heuristic
Learned Agent GreedyAgent + Learned Heuristic


Playing tournament:   0%|          | 0/50 [00:00<?, ?it/s]

Playing tournament:   0%|          | 0/50 [00:00<?, ?it/s]


AttributeError: 'GoState' object has no attribute 'observation_spec'